# 🐼 Day 3 — Pandas Deep Dive: Titanic Cleaning Pipeline

**Objective:** Build a reproducible 10-step data cleaning pipeline on the Titanic dataset.  
**Deliverable:** `data/titanic_clean.csv` — fully cleaned, validated, export-ready.  
**Topics covered:** Pandas 2.x CoW, method chaining, groupby imputation, Polars equivalents.

---
| Step | Operation |
|------|-----------|
| 1 | Inspect metadata |
| 2 | Standardize column names |
| 3 | Handle missing values |
| 4 | Convert dtypes |
| 5 | Fix inconsistent categories |
| 6 | Create derived features |
| 7 | Winsorize outliers |
| 8 | Encode categoricals |
| 9 | Validate integrity |
| 10 | Export + summary report |

## ⚙️ Environment Check

In [1]:
import sys
import subprocess
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)

print("=" * 50)
print("ENVIRONMENT")
print("=" * 50)
print(f"Python  : {sys.version.split()[0]}")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")

try:
    import polars as pl
    print(f"Polars  : {pl.__version__}")
except ImportError:
    print("Polars  : not installed  →  pip install polars")

try:
    import duckdb
    print(f"DuckDB  : {duckdb.__version__}")
except ImportError:
    print("DuckDB  : not installed (optional)")

try:
    res = subprocess.run(["nvidia-smi", "--query-gpu=name",
                          "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=5)
    gpu = res.stdout.strip() if res.returncode == 0 else "none"
except (FileNotFoundError, subprocess.TimeoutExpired):
    gpu = "none"
print(f"GPU     : {gpu or 'CPU only — fine for tabular work'}")
print("=" * 50)

# Verify Pandas 2.x CoW support
assert int(pd.__version__.split(".")[0]) >= 2, "Upgrade: pip install 'pandas>=2.0'"
print(f"\n✅ Pandas 2.x confirmed  (CoW available)")

ENVIRONMENT
Python  : 3.13.15
Pandas  : 2.3.3
NumPy   : 2.3.5


Polars  : 1.44.1


DuckDB  : 1.5.5
GPU     : none

✅ Pandas 2.x confirmed  (CoW available)


## 📥 Load Raw Data

We download the standard Kaggle Titanic training set directly from a public GitHub mirror.
No Kaggle credentials required.  Set `FORCE_DOWNLOAD = True` to re-download.

In [2]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

RAW_PATH   = DATA_DIR / "titanic.csv"
CLEAN_PATH = DATA_DIR / "titanic_clean.csv"

FORCE_DOWNLOAD = False   # ← set True to re-download

if not RAW_PATH.exists() or FORCE_DOWNLOAD:
    import urllib.request
    url = ("https://raw.githubusercontent.com/"
           "datasciencedojo/datasets/master/titanic.csv")
    print(f"Downloading from:\n  {url}")
    urllib.request.urlretrieve(url, RAW_PATH)
    print(f"✅ Saved to {RAW_PATH}")
else:
    print(f"✅ Using cached file: {RAW_PATH}")

df_raw = pd.read_csv(RAW_PATH)
print(f"\nShape  : {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()

✅ Using cached file: data\titanic.csv

Shape  : (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---
## Step 1 — Inspect Dataset & Metadata

**Why:** Understand shape, dtypes, missing rates, and value distributions *before* touching anything.  
**Pitfall:** Skipping this step leads to wrong imputation strategies — e.g., filling Age with a global mean instead of a group-conditional median.  
**Watch for:** High-cardinality columns (Ticket, Cabin) and heavy missingness (Cabin: 77%).

In [3]:
# ── Step 1: Inspect ──────────────────────────────────────────────────────
print("=" * 60)
print("1. DATASET INFO")
print("=" * 60)
df_raw.info()

print("\n── Missing values ──")
missing = df_raw.isnull().sum().rename("missing_count")
missing_pct = (missing / len(df_raw) * 100).rename("pct")
miss_summary = pd.concat([missing, missing_pct], axis=1)
print(miss_summary[miss_summary["missing_count"] > 0].to_string())

print("\n── Value counts (categorical columns) ──")
for col in ["Survived", "Pclass", "Sex", "Embarked"]:
    print(f"  {col}: {df_raw[col].value_counts().to_dict()}")

print("\n── Descriptive statistics ──")
display = __builtins__.__dict__.get("display", print)  # works in Jupyter and script
display(df_raw.describe(include="all"))

# ── Quick assertions ──
assert df_raw.shape == (891, 12), f"Expected (891,12), got {df_raw.shape}"
assert "Age" in df_raw.columns
print("\n✅ Step 1 complete — 891 rows × 12 columns confirmed")

1. DATASET INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

── Missing values ──
          missing_count        pct
Age                 177  19.865320
Cabin               687  77.104377
Embarked              2   0.224467

── Value counts (categorical columns) ──
  Survived: {0: 549, 1: 

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN



✅ Step 1 complete — 891 rows × 12 columns confirmed


---
## Step 2 — Standardize Column Names

**Why:** Snake_case names make method chaining cleaner and prevent attribute-access issues with spaces.  
**Pitfall:** PascalCase like `PassengerId` breaks `df.passenger_id` attribute access — standardize early so all downstream code is consistent.

In [4]:
# ── Step 2: Rename columns ────────────────────────────────────────────────
df = df_raw.copy()   # work on a clean copy; df_raw stays pristine for comparison

RENAME_MAP = {
    "PassengerId": "passenger_id",
    "Survived":    "survived",
    "Pclass":      "pclass",
    "Name":        "name",
    "Sex":         "sex",
    "Age":         "age",
    "SibSp":       "sibsp",
    "Parch":       "parch",
    "Ticket":      "ticket",
    "Fare":        "fare",
    "Cabin":       "cabin",
    "Embarked":    "embarked",
}

df = df.rename(columns=RENAME_MAP)

# Safety pass: lowercase anything remaining, replace spaces
df.columns = [c.lower().replace(" ", "_").replace("-", "_") for c in df.columns]

print("Columns after rename:")
print(df.columns.tolist())

# ── Assertions ──
assert list(df.columns) == list(RENAME_MAP.values()), \
    f"Rename mismatch. Got: {df.columns.tolist()}"
assert " " not in "".join(df.columns), "Column names still contain spaces"
print("\n✅ Step 2 complete — all column names are snake_case")

Columns after rename:
['passenger_id', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']

✅ Step 2 complete — all column names are snake_case


---
## Step 3 — Handle Missing Values

**Strategy per column:**
| Column | Missing | Strategy | Rationale |
|--------|---------|----------|-----------|
| `age` | 177 (19.9%) | Median by `pclass` × `sex` group | Age correlates with class and sex |
| `cabin` | 687 (77.1%) | Create `has_cabin` flag, drop original | Too sparse to impute; flag captures cabin-assignment signal |
| `embarked` | 2 (0.2%) | Fill with mode (`S`) | Near-zero missingness, safe to use most frequent |

**Pitfall:** Global mean/median imputation ignores group structure — imputing the same median age for a 3rd-class male child and a 1st-class female adult adds noise.

In [5]:
# ── Step 3: Handle missing values ─────────────────────────────────────────
print("Before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

# 3a. cabin → binary presence flag, drop original
df["has_cabin"] = df["cabin"].notna().astype(np.int8)
df = df.drop(columns=["cabin"])

# 3b. embarked → fill with mode
embarked_mode = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(embarked_mode)
print(f"\nEmbarked mode used for fill: '{embarked_mode}'")

# 3c. age → group-conditional median imputation (vectorised with groupby + transform)
age_group_median = df.groupby(["pclass", "sex"])["age"].transform("median")
df["age"] = df["age"].fillna(age_group_median)

# Fallback: global median if any still null (shouldn't happen on this dataset)
remaining_null_age = df["age"].isnull().sum()
if remaining_null_age:
    print(f"  ⚠ {remaining_null_age} age values still null — filling with global median")
    df["age"] = df["age"].fillna(df["age"].median())

print("\nAfter cleaning:")
print(df.isnull().sum().to_string())

# ── Assertions ──
assert df["age"].isnull().sum() == 0,      "Age still has nulls"
assert df["embarked"].isnull().sum() == 0, "Embarked still has nulls"
assert "cabin" not in df.columns,          "cabin should be dropped"
assert "has_cabin" in df.columns,          "has_cabin flag missing"
total_null = df.isnull().sum().sum()
print(f"\n✅ Step 3 complete — total remaining nulls: {total_null}")

Before cleaning:
age         177
cabin       687
embarked      2

Embarked mode used for fill: 'S'

After cleaning:


passenger_id    0
survived        0
pclass          0
name            0
sex             0
age             0
sibsp           0
parch           0
ticket          0
fare            0
embarked        0
has_cabin       0

✅ Step 3 complete — total remaining nulls: 0


---
## Step 4 — Convert Data Types

**Why:** Correct dtypes reduce memory by 30–70% and enable categorical operations.  
**Pitfall:** Converting `pclass` to `int` first then to ordered `Categorical` works; doing it in one shot from `object` sometimes fails — always check the current dtype before converting.

In [6]:
# ── Step 4: dtype conversion ──────────────────────────────────────────────
print("Before conversion:")
print(df.dtypes.to_string())

# Ordered categorical for class
df["pclass"] = pd.Categorical(df["pclass"].astype(int),
                               categories=[1, 2, 3], ordered=True)

# Unordered categoricals
df["sex"]      = df["sex"].astype("category")
df["embarked"] = df["embarked"].astype("category")

# Compact numerics (saves memory)
df["survived"] = df["survived"].astype(np.int8)
df["sibsp"]    = df["sibsp"].astype(np.int8)
df["parch"]    = df["parch"].astype(np.int8)
df["age"]      = df["age"].astype(np.float32)
df["fare"]     = df["fare"].astype(np.float32)

print("\nAfter conversion:")
print(df.dtypes.to_string())

mem_before = df_raw.memory_usage(deep=True).sum() / 1024
mem_after  = df.memory_usage(deep=True).sum() / 1024
print(f"\nMemory: {mem_before:.1f} KB → {mem_after:.1f} KB  "
      f"({100*(mem_before-mem_after)/mem_before:.0f}% reduction)")

# ── Assertions ──
assert df["pclass"].dtype.name == "category",      "pclass should be Categorical"
assert df["survived"].dtype == np.int8,            "survived should be int8"
assert df["age"].dtype == np.float32,              "age should be float32"
assert df["pclass"].cat.ordered,                   "pclass should be ordered"
print("\n✅ Step 4 complete — dtypes optimized")

Before conversion:
passenger_id      int64
survived          int64
pclass            int64
name             object
sex              object
age             float64
sibsp             int64
parch             int64
ticket           object
fare            float64
embarked         object
has_cabin          int8

After conversion:
passenger_id       int64
survived            int8
pclass          category
name              object
sex             category
age              float32
sibsp               int8
parch               int8
ticket            object
fare             float32
embarked        category
has_cabin           int8

Memory: 285.6 KB → 135.3 KB  (53% reduction)

✅ Step 4 complete — dtypes optimized


---
## Step 5 — Fix Inconsistent Categories & Typos

**Why:** Whitespace, casing differences (`Female` vs `female`), and mislabelled values cause silent groupby errors and wrong model encodings.  
**Pitfall:** `str.strip().str.lower()` on a `Categorical` column raises a TypeError in Pandas 2.x — convert to `str` first, then back to `Categorical`.

In [7]:
# ── Step 5: Normalize category values ─────────────────────────────────────

# Convert to str → clean → back to Categorical
df["sex"]      = df["sex"].astype(str).str.strip().str.lower()
df["embarked"] = df["embarked"].astype(str).str.strip().str.upper()

# Standardize: only expected values survive
VALID_SEX      = {"male", "female"}
VALID_EMBARKED = {"C", "Q", "S"}

# Replace anything unexpected with NaN, then assert 0 nulls
df["sex"]      = df["sex"].where(df["sex"].isin(VALID_SEX))
df["embarked"] = df["embarked"].where(df["embarked"].isin(VALID_EMBARKED))

# Convert back to Categorical
df["sex"]      = pd.Categorical(df["sex"])
df["embarked"] = pd.Categorical(df["embarked"])

print(f"sex      unique values: {sorted(df['sex'].unique().tolist())}")
print(f"embarked unique values: {sorted(df['embarked'].unique().tolist())}")
print(f"\nsex distribution:\n{df['sex'].value_counts().to_string()}")
print(f"\nembarked distribution:\n{df['embarked'].value_counts().to_string()}")

# ── Assertions ──
assert set(df["sex"].unique().tolist()) <= VALID_SEX,      \
    f"Unexpected sex values: {df['sex'].unique()}"
assert set(df["embarked"].unique().tolist()) <= VALID_EMBARKED, \
    f"Unexpected embarked values: {df['embarked'].unique()}"
assert df["sex"].isnull().sum() == 0,      "Nulls introduced in sex column"
assert df["embarked"].isnull().sum() == 0, "Nulls introduced in embarked column"
print("\n✅ Step 5 complete — all category values validated")

sex      unique values: ['female', 'male']
embarked unique values: ['C', 'Q', 'S']



sex distribution:
sex
male      577
female    314

embarked distribution:
embarked
S    646
C    168
Q     77

✅ Step 5 complete — all category values validated


---
## Step 6 — Create Derived Features

**Why:** Raw columns rarely capture domain knowledge directly. `Title` captures social status/age proxy; `family_size` captures traveling-group effects.  
**Pitfall:** The regex `r",\s*([^\.]+)\."` is greedy — test it with `df['name'].str.extract(...)` before applying to verify all names match.

In [8]:
# ── Step 6: Feature engineering ────────────────────────────────────────────

# 6a. Extract honorific title from name
#   Name format: "Braund, Mr. Owen Harris"
df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.").squeeze()

print("Raw title counts:")
print(df["title"].value_counts().to_string())

# Group rare/international titles
TITLE_GROUPS = {
    "Mr":       "Mr",
    "Miss":     "Miss",
    "Mrs":      "Mrs",
    "Master":   "Master",
    "Dr":       "Rare",
    "Rev":      "Rare",
    "Col":      "Rare",
    "Major":    "Rare",
    "Mlle":     "Miss",   # French equivalent of Miss
    "Mme":      "Mrs",    # French equivalent of Mrs
    "Ms":       "Miss",
    "Lady":     "Rare",
    "Jonkheer": "Rare",
    "Don":      "Rare",
    "Sir":      "Rare",
    "Capt":     "Rare",
    "the Countess": "Rare",
}

df["title_group"] = (df["title"].str.strip()
                               .map(TITLE_GROUPS)
                               .fillna("Rare"))
df["title_group"] = pd.Categorical(df["title_group"])

print("\nGrouped title counts:")
print(df["title_group"].value_counts().to_string())

# 6b. Family size: siblings/spouses + parents/children + self
df["family_size"] = df["sibsp"].astype(int) + df["parch"].astype(int) + 1

# 6c. Is alone
df["is_alone"] = (df["family_size"] == 1).astype(np.int8)

print(f"\nfamily_size range : {df['family_size'].min()} – {df['family_size'].max()}")
print(f"is_alone count    : {df['is_alone'].sum()} passengers traveling alone")

# ── Assertions ──
assert df["title"].isnull().sum() == 0,         "Regex failed to match some names"
assert df["title_group"].isnull().sum() == 0,   "Some titles not mapped"
assert (df["family_size"] >= 1).all(),           "family_size must be >= 1 (self)"
assert df["is_alone"].isin([0, 1]).all(),        "is_alone must be binary"
print("\n✅ Step 6 complete — title_group, family_size, is_alone created")

Raw title counts:
title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1

Grouped title counts:
title_group
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23

family_size range : 1 – 11
is_alone count    : 537 passengers traveling alone



✅ Step 6 complete — title_group, family_size, is_alone created


---
## Step 7 — Handle Outliers (Winsorize)

**Why:** The raw `fare` has extreme values (e.g., £512) that skew model gradients.  
Winsorizing caps values at a percentile rather than dropping rows, preserving observations.  
**Pitfall:** Cap *after* feature engineering (Step 6 already complete) — capping fare before deriving other features is fine here, but capping before group-median age imputation would distort the medians.

In [9]:
# ── Step 7: Winsorize outliers ─────────────────────────────────────────────

print("Before winsorizing:")
print(df[["age", "fare"]].describe().to_string())

FARE_CAP = float(df["fare"].quantile(0.99))
AGE_CAP  = float(df["age"].quantile(0.99))

df["fare"] = df["fare"].clip(upper=FARE_CAP).astype(np.float32)
df["age"]  = df["age"].clip(upper=AGE_CAP).astype(np.float32)

print("\nAfter winsorizing:")
print(df[["age", "fare"]].describe().to_string())
print(f"\nFare cap (99th pct)  : {FARE_CAP:.2f}")
print(f"Age cap  (99th pct)  : {AGE_CAP:.2f}")

# ── Assertions ──
assert float(df["fare"].max()) <= FARE_CAP + 0.01, "Fare cap not applied"
assert float(df["age"].max())  <= AGE_CAP  + 0.01, "Age cap not applied"
assert (df["fare"] >= 0).all(), "Negative fares found after capping"
print("\n✅ Step 7 complete — fare and age winsorized at 99th percentile")

Before winsorizing:
              age        fare
count  891.000000  891.000000
mean    29.112425   32.204208
std     13.304424   49.693432
min      0.420000    0.000000
25%     21.500000    7.910400
50%     26.000000   14.454200
75%     36.000000   31.000000
max     80.000000  512.329224

After winsorizing:
              age        fare
count  891.000000  891.000000
mean    29.053501   31.224766
std     13.124568   42.524124
min      0.420000    0.000000
25%     21.500000    7.910400
50%     26.000000   14.454200
75%     36.000000   31.000000
max     65.000000  249.006226

Fare cap (99th pct)  : 249.01
Age cap  (99th pct)  : 65.00

✅ Step 7 complete — fare and age winsorized at 99th percentile


---
## Step 8 — Encode Categorical Columns

**Strategy:**
| Column | Encoding | Reason |
|--------|----------|--------|
| `sex` | Binary (0/1) | Two values only |
| `embarked` | One-hot | 3 values, no ordinal relationship |
| `title_group` | One-hot | 5 values, no ordinal relationship |

**Pitfall:** Don't one-hot encode `pclass` — it's an *ordered* categorical and tree models can use ordinal form directly. For linear models, you would one-hot it too.

In [10]:
# ── Step 8: Encode categoricals ────────────────────────────────────────────

# 8a. sex → binary integer (female=0, male=1)
df["sex_male"] = (df["sex"] == "male").astype(np.int8)

# 8b. embarked → one-hot  (dtype int8 to stay memory-efficient)
emb_dummies = pd.get_dummies(df["embarked"], prefix="emb", dtype=np.int8)
df = pd.concat([df, emb_dummies], axis=1)

# 8c. title_group → one-hot
title_dummies = pd.get_dummies(df["title_group"], prefix="title", dtype=np.int8)
df = pd.concat([df, title_dummies], axis=1)

# 8d. Drop source columns now encoded; also drop high-cardinality text columns
df = df.drop(columns=["sex", "embarked", "title", "title_group",
                        "name", "ticket"])

print("Columns after encoding:")
for c in df.columns:
    print(f"  {c:<30} {df[c].dtype}")
print(f"\nShape: {df.shape}")

# ── Assertions ──
assert "sex" not in df.columns,         "raw sex column should be dropped"
assert "sex_male" in df.columns,        "sex_male encoding missing"
assert "emb_C" in df.columns,           "embarked one-hot missing"
assert "title_Mr" in df.columns,        "title one-hot missing"
assert df["sex_male"].isin([0,1]).all(), "sex_male should be binary"
print("\n✅ Step 8 complete — categoricals encoded, high-cardinality columns dropped")

Columns after encoding:
  passenger_id                   int64
  survived                       int8
  pclass                         category
  age                            float32
  sibsp                          int8
  parch                          int8
  fare                           float32
  has_cabin                      int8
  family_size                    int64
  is_alone                       int8
  sex_male                       int8
  emb_C                          int8
  emb_Q                          int8
  emb_S                          int8
  title_Master                   int8
  title_Miss                     int8
  title_Mr                       int8
  title_Mrs                      int8
  title_Rare                     int8

Shape: (891, 19)

✅ Step 8 complete — categoricals encoded, high-cardinality columns dropped


---
## Step 9 — Validate Data Integrity

**Why:** Bugs introduced in earlier steps surface here. Validation before export prevents silent corruption reaching the model training stage.  
**Pitfall:** Integrity checks on intermediate DataFrames are useless if the pipeline isn't run end-to-end — always rerun all cells (Kernel > Restart & Run All) before committing.

In [11]:
# ── Step 9: Validate data integrity ────────────────────────────────────────
print("=" * 60)
print("DATA INTEGRITY REPORT")
print("=" * 60)

FAILURES = []

# Check 1: No duplicate rows
n_dupes = df.duplicated().sum()
status = "✅" if n_dupes == 0 else f"❌ ({n_dupes} found)"
print(f"Duplicate rows   : {status}")
if n_dupes: FAILURES.append("Duplicates found")

# Check 2: No nulls anywhere
n_nulls = df.isnull().sum().sum()
status = "✅" if n_nulls == 0 else f"❌ ({n_nulls} total nulls)"
print(f"Total null values: {status}")
if n_nulls: FAILURES.append(f"{n_nulls} nulls remain")

# Check 3: survived is binary {0, 1}
bad_survived = ~df["survived"].isin([0, 1])
status = "✅" if not bad_survived.any() else f"❌ {bad_survived.sum()} bad values"
print(f"survived ∈ {{0,1}} : {status}")
if bad_survived.any(): FAILURES.append("survived has invalid values")

# Check 4: age in valid range
age_ok = df["age"].between(0, 100)
status = f"✅  [{df['age'].min():.1f}, {df['age'].max():.1f}]" if age_ok.all() else "❌"
print(f"age ∈ [0, 100]   : {status}")
if not age_ok.all(): FAILURES.append("age out of range")

# Check 5: fare non-negative
fare_ok = df["fare"] >= 0
status = f"✅  [{df['fare'].min():.2f}, {df['fare'].max():.2f}]" if fare_ok.all() else "❌"
print(f"fare >= 0        : {status}")
if not fare_ok.all(): FAILURES.append("negative fares found")

# Check 6: expected row count
status = f"✅  ({df.shape[0]} rows)" if df.shape[0] == 891 else f"❌ ({df.shape[0]} rows)"
print(f"Row count = 891  : {status}")
if df.shape[0] != 891: FAILURES.append(f"row count mismatch: {df.shape[0]}")

print("=" * 60)
if FAILURES:
    print(f"\n❌ {len(FAILURES)} checks FAILED:")
    for f in FAILURES:
        print(f"   • {f}")
    raise AssertionError("Integrity checks failed — fix pipeline before export")
else:
    print(f"\n✅ Step 9 complete — all 6 integrity checks passed")
    print(f"   Final shape: {df.shape}")

DATA INTEGRITY REPORT
Duplicate rows   : ✅
Total null values: ✅
survived ∈ {0,1} : ✅
age ∈ [0, 100]   : ✅  [0.4, 65.0]
fare >= 0        : ✅  [0.00, 249.01]
Row count = 891  : ✅  (891 rows)

✅ Step 9 complete — all 6 integrity checks passed
   Final shape: (891, 19)


---
## Step 10 — Export Cleaned Dataset & Summary Report

**Why:** Exporting early creates a reusable artifact so downstream ML notebooks don't rerun the pipeline.  
**Pitfall:** `index=False` is mandatory — including the default RangeIndex adds a meaningless unnamed column that confuses autoML tools and `pd.read_csv`.

In [12]:
# ── Step 10: Export + summary ──────────────────────────────────────────────
df.to_csv(CLEAN_PATH, index=False)

print(f"✅ Cleaned dataset exported to: {CLEAN_PATH}")
print(f"   File size : {CLEAN_PATH.stat().st_size / 1024:.1f} KB")
print(f"   Shape     : {df.shape}")

print("\n" + "=" * 60)
print("CLEANING SUMMARY REPORT")
print("=" * 60)
print(f"  Original shape : {df_raw.shape}")
print(f"  Final shape    : {df.shape}")
print(f"  Columns added  : has_cabin, family_size, is_alone, sex_male, "
      f"emb_*, title_*")
print(f"  Columns removed: name, ticket, cabin, sex, embarked, title, title_group")
print(f"  Nulls (before) : {df_raw.isnull().sum().sum()}")
print(f"  Nulls (after)  : {df.isnull().sum().sum()}")
print(f"  Survival rate  : {df['survived'].mean():.1%}")
print(f"  Avg age (clean): {df['age'].mean():.1f} yrs")
print(f"  Passengers alone: {df['is_alone'].sum()} ({df['is_alone'].mean():.1%})")
print("=" * 60)

print("\nSample rows (first 5):")
display = __builtins__.__dict__.get("display", print)
display(df.head())

✅ Cleaned dataset exported to: data\titanic_clean.csv
   File size : 41.9 KB
   Shape     : (891, 19)

CLEANING SUMMARY REPORT
  Original shape : (891, 12)
  Final shape    : (891, 19)
  Columns added  : has_cabin, family_size, is_alone, sex_male, emb_*, title_*
  Columns removed: name, ticket, cabin, sex, embarked, title, title_group
  Nulls (before) : 866
  Nulls (after)  : 0
  Survival rate  : 38.4%
  Avg age (clean): 29.1 yrs
  Passengers alone: 537 (60.3%)

Sample rows (first 5):


,passenger_id,survived,pclass,age,sibsp,parch,fare,has_cabin,family_size,is_alone,sex_male,emb_C,emb_Q,emb_S,title_Master,title_Miss,title_Mr,title_Mrs,title_Rare
0,1,0,3,22.0,1,0,7.250000,0,2,0,1,0,0,1,0,0,1,0,0
1,2,1,1,38.0,1,0,71.283302,1,2,0,0,1,0,0,0,0,0,1,0
2,3,1,3,26.0,0,0,7.925000,0,1,1,0,0,0,1,0,1,0,0,0
3,4,1,1,35.0,1,0,53.099998,1,2,0,0,0,0,1,0,0,0,1,0
4,5,0,3,35.0,0,0,8.050000,0,1,1,1,0,0,1,0,0,1,0,0


---
## 📋 Copy-on-Write (CoW) in Pandas 2.x

**Background:**  
Before Pandas 2.0, `df[mask]` might return a **view** or a **copy** depending on internal state — a notorious source of the dreaded `SettingWithCopyWarning`. Pandas 2.0 introduced opt-in CoW; Pandas 2.2+ enables it by default.

**CoW rule:** Any write operation on a DataFrame (or Series) derived from another always triggers a copy of the modified data — so the parent is never silently mutated.

**Three scenarios every practitioner must know:**

In [13]:
# ── Copy-on-Write demonstration ─────────────────────────────────────────────
import pandas as pd, numpy as np

print(f"Pandas version         : {pd.__version__}")

# Enable CoW explicitly for this demo (no-op if already default in 2.2+)
pd.options.mode.copy_on_write = True
print(f"copy_on_write enabled  : {pd.options.mode.copy_on_write}")

demo = pd.DataFrame({
    "pclass": [1, 1, 2, 3, 3],
    "age":    [30., 45., 28., 22., 18.],
    "fare":   [211., 263., 13., 7., 8.],
})

# ── Scenario 1: Filter then modify — CoW protects the parent ──────────────
print("\n─── Scenario 1: Filter → modify (CoW protects parent) ───")
subset = demo[demo["pclass"] == 1]   # returns a new CoW-backed object
subset.loc[:, "age"] = 0            # modifies subset only; demo unchanged
print(f"demo['age'] still intact: {demo['age'].tolist()}")   # [30, 45, 28, 22, 18]
print("→ With CoW: subset modification does NOT propagate to demo ✅")

# ── Scenario 2: Explicit .copy() — always portable, pre/post CoW ──────────
print("\n─── Scenario 2: Explicit .copy() (portable across all Pandas versions) ───")
subset2 = demo[demo["pclass"] == 1].copy()   # guaranteed independent copy
subset2.loc[:, "age"] = -1
print(f"demo['age']   : {demo['age'].tolist()}")    # unchanged
print(f"subset2['age']: {subset2['age'].tolist()}")  # [-1, -1]
print("→ .copy() is explicit, readable, and safe in any Pandas version ✅")

# ── Scenario 3: In-place update on parent via .loc — always works ─────────
print("\n─── Scenario 3: In-place update with .loc on parent ───")
demo2 = demo.copy()
demo2.loc[demo2["pclass"] == 1, "age"] = 99   # modify parent directly
print(f"demo2 (pclass==1 ages): {demo2.loc[demo2['pclass']==1,'age'].tolist()}")
print("→ .loc on the original DataFrame modifies it in-place — always intentional ✅")

# ── Rule of thumb ─────────────────────────────────────────────────────────
print("\n─── Rule of thumb ───")
print("  READING  : any indexing is safe (CoW ensures lazy copy-on-write)")
print("  WRITING  : always use .loc on the original, or call .copy() first")
print("  AVOID    : chained assignment  e.g.  df[mask]['col'] = value  (never reliable)")

pd.options.mode.copy_on_write = False   # reset to default for rest of notebook

Pandas version         : 2.3.3
copy_on_write enabled  : True

─── Scenario 1: Filter → modify (CoW protects parent) ───
demo['age'] still intact: [30.0, 45.0, 28.0, 22.0, 18.0]
→ With CoW: subset modification does NOT propagate to demo ✅

─── Scenario 2: Explicit .copy() (portable across all Pandas versions) ───
demo['age']   : [30.0, 45.0, 28.0, 22.0, 18.0]
subset2['age']: [-1.0, -1.0]
→ .copy() is explicit, readable, and safe in any Pandas version ✅

─── Scenario 3: In-place update with .loc on parent ───
demo2 (pclass==1 ages): [99.0, 99.0]
→ .loc on the original DataFrame modifies it in-place — always intentional ✅

─── Rule of thumb ───
  READING  : any indexing is safe (CoW ensures lazy copy-on-write)
  WRITING  : always use .loc on the original, or call .copy() first
  AVOID    : chained assignment  e.g.  df[mask]['col'] = value  (never reliable)


---
## ⚡ Polars Equivalents (3 Cleaning Steps)

Polars is a Rust-based DataFrame library that executes queries multi-threaded and uses lazy evaluation.  
Key differences:
- **Immutable by default:** every operation returns a new DataFrame (no in-place mutations).
- **Lazy API:** `scan_csv` + `collect()` builds a query plan before executing — allows filter pushdown and predicate optimization.
- **No index:** Polars has no row index; this removes a whole class of alignment bugs.

In [14]:
try:
    import polars as pl
    print(f"Polars {pl.__version__} available ✅")
except ImportError:
    print("Polars not installed — run: pip install polars")
    raise SystemExit

DATA_DIR = __import__("pathlib").Path("data")

# ────────────────────────────────────────────────────────────────────────
# Polars Step 2 equivalent: Rename columns
# ────────────────────────────────────────────────────────────────────────
print("\n── Polars Step 2: Rename columns ──")

RENAME_PL = {
    "PassengerId": "passenger_id", "Survived": "survived",
    "Pclass": "pclass",            "Name": "name",
    "Sex": "sex",                  "Age": "age",
    "SibSp": "sibsp",              "Parch": "parch",
    "Ticket": "ticket",            "Fare": "fare",
    "Cabin": "cabin",              "Embarked": "embarked",
}

df_pl = pl.read_csv(DATA_DIR / "titanic.csv", null_values=[""]).rename(RENAME_PL)
print(f"Columns: {df_pl.columns}")
print(f"Shape  : {df_pl.shape}")
# Performance note: rename is zero-copy in Polars (only metadata changes).

# ────────────────────────────────────────────────────────────────────────
# Polars Step 3 equivalent: Handle missing values
# ────────────────────────────────────────────────────────────────────────
print("\n── Polars Step 3: Missing value imputation ──")

median_age = df_pl["age"].median()
mode_embarked = df_pl["embarked"].drop_nulls().mode()[0]

df_pl = (
    df_pl
    .with_columns(
        pl.col("cabin").is_not_null().cast(pl.Int8).alias("has_cabin"),
        pl.col("age").fill_null(median_age),
        pl.col("embarked").fill_null(mode_embarked),
    )
    .drop("cabin")
)

print(f"age nulls    : {df_pl['age'].null_count()}")
print(f"embarked nulls: {df_pl['embarked'].null_count()}")
# Performance note: with_columns is lazy-composable; all expressions evaluate in parallel.

# ────────────────────────────────────────────────────────────────────────
# Polars Step 6 equivalent: Derived features
# ────────────────────────────────────────────────────────────────────────
print("\n── Polars Step 6: Derived features ──")

df_pl = df_pl.with_columns([
    (pl.col("sibsp") + pl.col("parch") + 1).alias("family_size"),
    pl.col("name").str.extract(r",\s*([^\.]+)\.", 1).alias("title"),
])
df_pl = df_pl.with_columns(
    (pl.col("family_size") == 1).cast(pl.Int8).alias("is_alone")
)

print("Sample (family features):")
print(df_pl.select(["passenger_id", "name", "title",
                     "family_size", "is_alone"]).head(5))

# ────────────────────────────────────────────────────────────────────────
# Polars Lazy API: query optimization example
# ────────────────────────────────────────────────────────────────────────
print("\n── Polars Lazy API: survivors by class ──")

result = (
    pl.scan_csv(str(DATA_DIR / "titanic.csv"), null_values=[""])
    .rename(RENAME_PL)
    .with_columns(
        (pl.col("sibsp") + pl.col("parch") + 1).alias("family_size")
    )
    .filter(pl.col("survived") == 1)
    .group_by("pclass")
    .agg(
        pl.col("age").mean().round(1).alias("avg_age"),
        pl.len().alias("count"),
        pl.col("fare").median().round(2).alias("median_fare"),
    )
    .sort("pclass")
    .collect()   # ← only now does execution happen
)
print(result)
# Performance note: scan_csv + collect allows Polars to push the filter
# and projection into the CSV read, skipping unused rows and columns.
print("\n✅ Polars equivalents complete")

Polars 1.44.1 available ✅

── Polars Step 2: Rename columns ──


Columns: ['passenger_id', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']
Shape  : (891, 12)

── Polars Step 3: Missing value imputation ──


age nulls    : 0
embarked nulls: 0

── Polars Step 6: Derived features ──
Sample (family features):
shape: (5, 5)
┌──────────────┬─────────────────────────────────┬───────┬─────────────┬──────────┐
│ passenger_id ┆ name                            ┆ title ┆ family_size ┆ is_alone │
│ ---          ┆ ---                             ┆ ---   ┆ ---         ┆ ---      │
│ i64          ┆ str                             ┆ str   ┆ i64         ┆ i8       │
╞══════════════╪═════════════════════════════════╪═══════╪═════════════╪══════════╡
│ 1            ┆ Braund, Mr. Owen Harris         ┆ Mr    ┆ 2           ┆ 0        │
│ 2            ┆ Cumings, Mrs. John Bradley (Fl… ┆ Mrs   ┆ 2           ┆ 0        │
│ 3            ┆ Heikkinen, Miss. Laina          ┆ Miss  ┆ 1           ┆ 1        │
│ 4            ┆ Futrelle, Mrs. Jacques Heath (… ┆ Mrs   ┆ 2           ┆ 0        │
│ 5            ┆ Allen, Mr. William Henry        ┆ Mr    ┆ 1           ┆ 1        │
└──────────────┴──────────────────────────────

shape: (3, 4)
┌────────┬─────────┬───────┬─────────────┐
│ pclass ┆ avg_age ┆ count ┆ median_fare │
│ ---    ┆ ---     ┆ ---   ┆ ---         │
│ i64    ┆ f64     ┆ u32   ┆ f64         │
╞════════╪═════════╪═══════╪═════════════╡
│ 1      ┆ 35.4    ┆ 136   ┆ 77.96       │
│ 2      ┆ 25.9    ┆ 87    ┆ 21.0        │
│ 3      ┆ 20.6    ┆ 119   ┆ 8.52        │
└────────┴─────────┴───────┴─────────────┘

✅ Polars equivalents complete


---
## 🚀 Final Checklist & Git Push

In [15]:
# ── Day 3 Deliverable Checklist ────────────────────────────────────────────
DATA_DIR   = __import__("pathlib").Path("data")
CLEAN_PATH = DATA_DIR / "titanic_clean.csv"

checks = [
    ("titanic_clean.csv exported",        CLEAN_PATH.exists()),
    ("titanic_clean.csv non-empty",        CLEAN_PATH.exists() and
                                           CLEAN_PATH.stat().st_size > 10_000),
]

import pandas as pd
if CLEAN_PATH.exists():
    _df = pd.read_csv(CLEAN_PATH)
    checks += [
        ("891 rows in clean CSV",              len(_df) == 891),
        ("zero nulls in clean CSV",            _df.isnull().sum().sum() == 0),
        ("sex_male column present",            "sex_male" in _df.columns),
        ("family_size column present",         "family_size" in _df.columns),
    ]

print("=" * 55)
print("DAY 3 AUTOMATED CHECKLIST")
print("=" * 55)
all_ok = True
for label, result in checks:
    icon = "✅" if result else "❌"
    print(f"  {icon}  {label}")
    if not result: all_ok = False

print("\nMANUAL CHECKLIST (verify yourself):")
manual = [
    "Notebook runs end-to-end (Kernel > Restart & Run All)",
    "Polars equivalents cells executed without error",
    "CoW behavior demo ran and output is correct",
    "GitHub repo pushed (see commands below)",
    "README.md updated with Day 3 description",
]
for item in manual:
    print(f"  ☐  {item}")

print("\n" + "=" * 55)
print("GIT PUSH COMMANDS (run in terminal from repo root)")
print("=" * 55)
print("""
  git add day3_pandas/
  git commit -m "feat: Day 3 — Titanic cleaning pipeline, CoW demo, Polars equivalents ✅"
  git push origin main
""")

DAY 3 AUTOMATED CHECKLIST
  ✅  titanic_clean.csv exported
  ✅  titanic_clean.csv non-empty
  ✅  891 rows in clean CSV
  ✅  zero nulls in clean CSV
  ✅  sex_male column present
  ✅  family_size column present

MANUAL CHECKLIST (verify yourself):
  ☐  Notebook runs end-to-end (Kernel > Restart & Run All)
  ☐  Polars equivalents cells executed without error
  ☐  CoW behavior demo ran and output is correct
  ☐  GitHub repo pushed (see commands below)
  ☐  README.md updated with Day 3 description

GIT PUSH COMMANDS (run in terminal from repo root)

  git add day3_pandas/
  git commit -m "feat: Day 3 — Titanic cleaning pipeline, CoW demo, Polars equivalents ✅"
  git push origin main

